# Themes v0 — minimal

Goal: a **ranked table** of news themes labelled `aligned` / `novel` against ICB Sectors.

Pipeline:

1. ICB Subsector definitions → embedding → **Sector centroids** (45)
2. Sample one week of headlines → strip wire prefixes & dates → **FinLang embeddings**
3. **BERTopic** on the embeddings
4. For each theme: cosine to every Sector centroid → **nearest sector + max cosine + bucket**
5. **Drill-down HTML** with top headlines per theme

Out of scope (deferred to v1+): rolling windows / persistence, ticker tagging, news-volume slope, 2D map.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import torch
import umap
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "raw_news_2025.csv"
ICB_PATH = PROJECT_ROOT / "notebooks" / "input" / "icb-structure-and-definitions.xlsx"
OUTPUT_HTML = PROJECT_ROOT / "notebooks" / "output" / "themes_v0.html"
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
RANDOM_SEED = 42

DATE_START = "2025-01-01T00:00:00+00:00"
DATE_END = "2025-01-08T00:00:00+00:00"  # 1 week
SAMPLE_N = 10_000                        # cap for fast iteration

MIN_TOPIC_SIZE = 60
MIN_SAMPLES = 10
TAU_COV = 0.30                           # cos >= TAU_COV → "aligned"
TAU_SHOW = 0.30                          # min cosine for a Sankey theme→sector link
TOP_K_SECTORS = 3                        # max sector links per theme (cross-sector cap)

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}")

Device: mps


## 1. ICB Sector centroids

Embed each Subsector (`name + definition`) with FinLang, then average per ICB Sector → 45 normalized vectors.

In [2]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

icb_raw = pd.read_excel(ICB_PATH, sheet_name="Mappable", header=1).dropna(subset=["Subsector"])
icb = pd.DataFrame(
    {
        "industry": icb_raw["Industry"].str.strip(),
        "supersector": icb_raw["Supersector"].str.strip(),
        "sector": icb_raw["Sector"].str.strip(),
        "subsector": icb_raw["Subsector"].str.strip(),
        "definition": icb_raw["Definition"].fillna("").str.strip(),
    }
).reset_index(drop=True)

sub_texts = (icb["subsector"] + ". " + icb["definition"]).tolist()
sub_emb = embedder.encode(
    sub_texts, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)

sector_meta = (
    icb.groupby("sector", as_index=False)
    .agg(industry=("industry", "first"), n_subsectors=("subsector", "count"))
    .sort_values(["industry", "sector"])  # group sectors by Industry for the Sankey
    .reset_index(drop=True)
)
sector_emb = np.stack(
    [sub_emb[(icb["sector"] == s).to_numpy()].mean(axis=0) for s in sector_meta["sector"]]
)
sector_emb /= np.linalg.norm(sector_emb, axis=1, keepdims=True)
print(f"Sectors: {len(sector_meta)}  (centroids of {len(icb)} subsectors)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Sectors: 45  (centroids of 173 subsectors)


## 2. News sample + preprocessing

Load one week of headlines (dedupe by text) → uniform random sample → strip wire prefixes (`BSECorpAnn: …`) and dates.

In [3]:
DATE_PATTERNS = (
    r"\b\d{4}[/\-]\d{1,2}[/\-]\d{1,2}\b",
    r"\b\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b",
    r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b",
    r"\bq[1-4]\s+\d{4}\b",
)


def strip_colon_prefix(text: str, run:bool=False) -> str:
    """Drop short wire-source prefixes like 'BSECorpAnn:', 'Dawn News.pk:', 'Assam Sentinel:'.

    Only strips when the prefix is short (≤ 30 chars, ≤ 4 words) so we don't
    accidentally chop real content like 'S. Korea's FSS Refers Korea Zinc Execs to Prosecutors: Yonhap'.
    Applied twice to handle stacked prefixes like 'Reuters: NDTV: …'.
    """
    if not run:
        return text
    for _ in range(2):
        if not text or ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        prefix, rest = prefix.strip(), rest.strip()
        if not prefix or not prefix[0].isalpha():
            return text
        if len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        if not rest:
            return text
        text = rest
    return text


def strip_dates(text: str) -> str:
    for pat in DATE_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)
    text = re.sub(r":\s*\d+\b", ":", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r":\s*$", "", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_headline(text: str) -> str:
    return strip_dates(strip_colon_prefix(text, run=False))


corpus = (
    pl.scan_csv(NEWS_PATH)
    .select(["Headline", "CaptureTime"])
    .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC"))
    .filter(
        (pl.col("CaptureTime") >= pl.lit(DATE_START).str.to_datetime(time_zone="UTC"))
        & (pl.col("CaptureTime") < pl.lit(DATE_END).str.to_datetime(time_zone="UTC"))
        & pl.col("Headline").is_not_null()
        & (pl.col("Headline").str.len_chars() > 0)
    )
    .unique(subset=["Headline"])
    .collect()
)
print(f"Unique headlines in window: {corpus.height:,}")

if corpus.height > SAMPLE_N:
    corpus = corpus.sample(n=SAMPLE_N, seed=RANDOM_SEED, shuffle=True)

raw_headlines = corpus["Headline"].to_list()
norm_headlines = [normalize_headline(h) for h in raw_headlines]
keep = [i for i, t in enumerate(norm_headlines) if t.strip()]
raw_headlines = [raw_headlines[i] for i in keep]
norm_headlines = [norm_headlines[i] for i in keep]
print(f"Headlines used:             {len(norm_headlines):,}\n")
print("Raw vs normalized:")
for r, n in zip(raw_headlines[:3], norm_headlines[:3]):
    print(f"  raw:  {r}")
    print(f"  norm: {n}\n")

Unique headlines in window: 151,122
Headlines used:             10,000

Raw vs normalized:
  raw:  UNB: Top 10 Indian Blockbusters of 2024: Record-Breaking Hits Ruling the Box Office
  norm: UNB: Top Indian Blockbusters of : Record-Breaking Hits Ruling the Box Office

  raw:  Daily FT: SJB calls on Govt. to revoke agreement with Adani
  norm: Daily FT: SJB calls on Govt. to revoke agreement with Adani

  raw:  Hi India: Assam offers lucrative investment opportunities in chips, tourism and other sectors: CM
  norm: Hi India: Assam offers lucrative investment opportunities in chips, tourism and other sectors: CM



## 3. Embed + BERTopic

In [4]:
news_emb = embedder.encode(
    norm_headlines, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
print(f"News embeddings: {news_emb.shape}")

bertopic_vectorizer = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    min_df=2, max_df=0.9,
)
bertopic_umap = umap.UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric="cosine", random_state=RANDOM_SEED,
)
bertopic_hdbscan = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE, min_samples=MIN_SAMPLES,
    metric="euclidean", cluster_selection_method="eom", prediction_data=True,
)
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=bertopic_umap,
    hdbscan_model=bertopic_hdbscan,
    vectorizer_model=bertopic_vectorizer,
    verbose=False,
)
topics, _ = topic_model.fit_transform(norm_headlines, news_emb)
topics_arr = np.array(topics)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f"BERTopic topics (excl. -1): {n_topics}")
print(f"Outliers:                   {(topics_arr == -1).mean():.1%}")

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

News embeddings: (10000, 768)
BERTopic topics (excl. -1): 40
Outliers:                   28.4%


## 4. Ranked table — theme × sector

For each theme: normalized centroid of its headline embeddings, max cosine to the 45 Sector centroids, and bucket (`aligned` if `max_cos ≥ TAU_COV`, else `novel`). Sorted by `max_cos` descending — read top-to-bottom from "most sector-aligned" to "most novel".

In [5]:
def keywords(topic_id: int, n_words: int = 6) -> str:
    return ", ".join(w for w, _ in (topic_model.get_topic(topic_id) or [])[:n_words])


valid_topics = sorted(t for t in set(topics) if t != -1)
centroids = []
rows = []
for t in valid_topics:
    mask = topics_arr == t
    c = news_emb[mask].mean(axis=0)
    c /= np.linalg.norm(c) + 1e-12
    centroids.append(c)
    rows.append({"topic": t, "n_docs": int(mask.sum()), "keywords": keywords(t)})
centroids = np.stack(centroids)

sim = centroids @ sector_emb.T              # (n_topics, 45)
nearest = sim.argmax(axis=1)

themes = pd.DataFrame(rows)
themes["max_cos"] = sim.max(axis=1).round(3)
themes["nearest_sector"] = sector_meta["sector"].to_numpy()[nearest]
themes["nearest_industry"] = sector_meta["industry"].to_numpy()[nearest]
themes["bucket"] = np.where(themes["max_cos"] >= TAU_COV, "aligned", "novel")
themes = themes.sort_values("max_cos", ascending=False).reset_index(drop=True)

print(
    f"aligned: {(themes['bucket'] == 'aligned').sum()}   "
    f"novel: {(themes['bucket'] == 'novel').sum()}\n"
)
themes[["topic", "n_docs", "bucket", "nearest_sector", "max_cos", "keywords"]]

aligned: 22   novel: 18



,topic,n_docs,bucket,nearest_sector,max_cos,keywords
0,32,77,aligned,"Oil, Gas and Coal",0.529,"saudi, oil, offshore, lng, crude, gas"
1,39,66,aligned,Real Estate Investment Trusts,0.521,"fund, trust, funds, csrs, income, etf"
2,34,72,aligned,"Gas, Water and Multi-utilities",0.520,"ferc, hydro, submits, electric, gas, power"
3,23,106,aligned,Finance and Credit Services,0.517,"bank, mortgage, trust, bancorp, financial, fwp"
4,25,88,aligned,Precious Metals and Mining,0.513,"gold, resources, mining, newsfile, minerals, s..."
5,15,161,aligned,Finance and Credit Services,0.504,"corp, technologies, international, pos, kohl, ..."
6,8,227,aligned,Automobiles and Parts,0.499,"sales, dec, tesla, vehicles, car, hyundai"
7,33,74,aligned,"Gas, Water and Multi-utilities",0.494,"solar, energy, solarquarter, plant, renewable,..."
8,7,239,aligned,Medical Equipment and Services,0.465,"therapeutics, cancer, trial, clinical, pharmac..."
9,12,187,aligned,"Gas, Water and Multi-utilities",0.462,"report, reports, results, quarterly, quarter, ..."


## 5. Sankey + drill-down HTML

**Sankey — 5 columns** following the full ICB cascade:

```
Themes ─► Subsector ─► Sector ─► Supersector ─► Industry
```

- **Assignment unit is the theme (BERTopic cluster), not individual headlines.** Headlines are grouped into themes first; we never assign a single news item to one subsector.
- Cosine matching is at **Sector** level. Keep sectors with `cos ≥ TAU_SHOW`, ranked by cosine, **top `TOP_K_SECTORS` per theme**. For each kept sector, pick the **best subsector** within it (argmax cosine). Cross-sector themes show up to 3 paths; single-sector themes show 1.
- Right-side cascade is deterministic (each subsector belongs to one sector → one supersector → one industry). Link width = `sector_cos × n_docs` propagated up the tree.
- Themes with no sector match flow into a `(no sector match)` sink.

**Drill-down** below the chart: top 10 headlines per theme.

In [6]:
import html as html_lib
import json as _json

TOP_HEADLINES_PER_THEME = 10   # for the drill-down list
N_PEEK = 3                     # headlines previewed in Sankey hovers


def _short(h: str, n: int = 90) -> str:
    h = h.strip()
    return html_lib.escape(h if len(h) <= n else h[:n] + "...")

# --- ICB hierarchical lookups ---
sub_to_sector = dict(zip(icb["subsector"], icb["sector"]))
sub_to_supersector = dict(zip(icb["subsector"], icb["supersector"]))
sub_to_industry = dict(zip(icb["subsector"], icb["industry"]))
sector_to_supersector = dict(zip(icb["sector"], icb["supersector"]))
sector_to_industry = dict(zip(icb["sector"], icb["industry"]))
ss_to_industry = dict(zip(icb["supersector"], icb["industry"]))

# Indices of subsector rows per sector (for argmax-within-sector lookup)
sector_to_sub_rows = {}
for k, row in icb.iterrows():
    sector_to_sub_rows.setdefault(row["sector"], []).append(k)

# Industry palette
INDUSTRY_PALETTE = px.colors.qualitative.Bold + px.colors.qualitative.Set3
industries_all = sorted(icb["industry"].unique())
industry_color = {ind: INDUSTRY_PALETTE[i % len(INDUSTRY_PALETTE)] for i, ind in enumerate(industries_all)}

sector_list_all = sector_meta["sector"].tolist()  # 45, grouped by Industry

# --- Top headlines per theme (used in hovers + drill-down) ---
sim_to_centroid = news_emb @ centroids.T   # (N_headlines, n_themes)
theme_top_norm, theme_top_idx = {}, {}
for t in valid_topics:
    col = valid_topics.index(t)
    in_topic = np.where(topics_arr == t)[0]
    scores = sim_to_centroid[in_topic, col]
    order = in_topic[np.argsort(-scores)[:max(N_PEEK, TOP_HEADLINES_PER_THEME)]]
    theme_top_idx[t] = order
    theme_top_norm[t] = [norm_headlines[i] for i in order]

# --- Edges: theme → best subsector (within each touched sector) ---
theme_edges = []          # (row_idx, topic_id, sub_name, sector_name, value, bucket, sector_cos)
novel_theme_edges = []    # (row_idx, topic_id, n_docs)

for i, row in themes.iterrows():
    t = int(row["topic"])
    topic_idx = valid_topics.index(t)
    cos_row = sim[topic_idx]
    above = [(j, c) for j, c in enumerate(cos_row) if c >= TAU_SHOW]
    above.sort(key=lambda x: -x[1])
    above = above[:TOP_K_SECTORS]
    if not above:
        novel_theme_edges.append((i, t, float(row["n_docs"])))
        continue
    theme_vec = centroids[topic_idx]
    for j, sector_cos in above:
        sector_name = sector_list_all[j]
        sub_rows = sector_to_sub_rows[sector_name]
        sub_cos_within = sub_emb[sub_rows] @ theme_vec
        best = sub_rows[int(np.argmax(sub_cos_within))]
        sub_name = icb["subsector"].iloc[best]
        w = float(sector_cos * row["n_docs"])
        theme_edges.append((i, t, sub_name, sector_name, w, row["bucket"], float(sector_cos)))

# --- Aggregate flows up the hierarchy ---
sub_flow, sec_flow, ss_flow, ind_flow = {}, {}, {}, {}
for _, _t, sub_name, sector, w, *_ in theme_edges:
    sub_flow[sub_name] = sub_flow.get(sub_name, 0.0) + w
for s, w in sub_flow.items():
    sec_flow[sub_to_sector[s]] = sec_flow.get(sub_to_sector[s], 0.0) + w
for s, w in sec_flow.items():
    ss_flow[sector_to_supersector[s]] = ss_flow.get(sector_to_supersector[s], 0.0) + w
for ss, w in ss_flow.items():
    ind_flow[ss_to_industry[ss]] = ind_flow.get(ss_to_industry[ss], 0.0) + w

# Sort each level by hierarchical key for visual coherence
active_sub = sorted(sub_flow.keys(),
                    key=lambda s: (sub_to_industry[s], sub_to_supersector[s], sub_to_sector[s], s))
active_sec = sorted(sec_flow.keys(),
                    key=lambda s: (sector_to_industry[s], sector_to_supersector[s], s))
active_ss  = sorted(ss_flow.keys(),
                    key=lambda ss: (ss_to_industry[ss], ss))
active_ind = sorted(ind_flow.keys())

# --- Theme labels & rich hovers ---
theme_labels, theme_hover = [], []
for _, r in themes.iterrows():
    t = int(r["topic"])
    kws = [k.strip() for k in (r["keywords"] or "").split(",") if k.strip()][:5]
    theme_labels.append(f"T{t}: {', '.join(kws)[:55]}")
    peek = "<br>".join(f"&bull; {_short(h)}" for h in theme_top_norm[t][:N_PEEK])
    theme_hover.append(
        f"<b>T{t}</b> &middot; <i>{r['bucket']}</i><br>"
        f"n_docs: {r['n_docs']:,}<br>"
        f"nearest sector: {r['nearest_sector']} (cos={r['max_cos']:.2f})<br>"
        f"keywords: {r['keywords']}<br><br>"
        f"<b>top headlines:</b><br>{peek}"
    )

# --- Build node lists ---
node_labels = (
    theme_labels + active_sub + active_sec + active_ss + active_ind + ["(no sector match)"]
)
node_colors = (
    ["#1a7f37" if b == "aligned" else "#bf5700" for b in themes["bucket"]]
    + [industry_color[sub_to_industry[s]] for s in active_sub]
    + [industry_color[sector_to_industry[s]] for s in active_sec]
    + [industry_color[ss_to_industry[ss]]   for ss in active_ss]
    + [industry_color[ind]                  for ind in active_ind]
    + ["#999"]
)
node_customdata = (
    theme_hover
    + [f"<b>Subsector</b>: {s}<br>Sector: {sub_to_sector[s]}<br>"
       f"Supersector: {sub_to_supersector[s]}<br>Industry: {sub_to_industry[s]}<br>"
       f"flow: {sub_flow[s]:.0f}"
       for s in active_sub]
    + [f"<b>Sector</b>: {s}<br>Supersector: {sector_to_supersector[s]}<br>"
       f"Industry: {sector_to_industry[s]}<br>flow: {sec_flow[s]:.0f}"
       for s in active_sec]
    + [f"<b>Supersector</b>: {ss}<br>Industry: {ss_to_industry[ss]}<br>"
       f"flow: {ss_flow[ss]:.0f}"
       for ss in active_ss]
    + [f"<b>Industry</b>: {ind}<br>flow: {ind_flow[ind]:.0f}" for ind in active_ind]
    + ["No sector match above τ_show"]
)

n_themes = len(theme_labels)
offset_sub = n_themes
offset_sec = offset_sub + len(active_sub)
offset_ss  = offset_sec + len(active_sec)
offset_ind = offset_ss  + len(active_ss)
novel_idx  = offset_ind + len(active_ind)

idx_sub = {s: offset_sub + i for i, s in enumerate(active_sub)}
idx_sec = {s: offset_sec + i for i, s in enumerate(active_sec)}
idx_ss  = {s: offset_ss  + i for i, s in enumerate(active_ss)}
idx_ind = {s: offset_ind + i for i, s in enumerate(active_ind)}

# --- Links ---
sources, targets, values, link_colors, link_hover = [], [], [], [], []

link_kinds, link_hi_colors = [], []

for theme_i, t, sub_name, sector, w, bucket, scos in theme_edges:
    sources.append(theme_i)
    targets.append(idx_sub[sub_name])
    values.append(w)
    link_colors.append("rgba(26,127,55,0.30)" if bucket == "aligned" else "rgba(191,87,0,0.30)")
    link_hi_colors.append("rgba(26,127,55,0.90)" if bucket == "aligned" else "rgba(191,87,0,0.90)")
    link_kinds.append("theme")
    peek = "<br>".join(f"&bull; {_short(h)}" for h in theme_top_norm[t][:N_PEEK])
    link_hover.append(
        f"{theme_labels[theme_i]}<br>&rarr; {sub_name}<br>"
        f"within {sector} &middot; cos={scos:.2f}<br><br>"
        f"<b>top headlines:</b><br>{peek}"
    )

for theme_i, t, w in novel_theme_edges:
    sources.append(theme_i)
    targets.append(novel_idx)
    values.append(w)
    link_colors.append("rgba(170,170,170,0.30)")
    link_hi_colors.append("rgba(120,120,120,0.90)")
    link_kinds.append("novel")
    peek = "<br>".join(f"&bull; {_short(h)}" for h in theme_top_norm[t][:N_PEEK])
    link_hover.append(
        f"{theme_labels[theme_i]}<br>no sector &ge; {TAU_SHOW}<br><br>"
        f"<b>top headlines:</b><br>{peek}"
    )

CASCADE_RGBA = "rgba(150,150,150,0.18)"
CASCADE_HI = "rgba(70,70,70,0.85)"
for s, w in sub_flow.items():
    sources.append(idx_sub[s]); targets.append(idx_sec[sub_to_sector[s]])
    values.append(w); link_colors.append(CASCADE_RGBA); link_hi_colors.append(CASCADE_HI); link_kinds.append("cascade")
    link_hover.append(f"{s} &rarr; {sub_to_sector[s]}")
for s, w in sec_flow.items():
    sources.append(idx_sec[s]); targets.append(idx_ss[sector_to_supersector[s]])
    values.append(w); link_colors.append(CASCADE_RGBA); link_hi_colors.append(CASCADE_HI); link_kinds.append("cascade")
    link_hover.append(f"{s} &rarr; {sector_to_supersector[s]}")
for ss, w in ss_flow.items():
    sources.append(idx_ss[ss]); targets.append(idx_ind[ss_to_industry[ss]])
    values.append(w); link_colors.append(CASCADE_RGBA); link_hi_colors.append(CASCADE_HI); link_kinds.append("cascade")
    link_hover.append(f"{ss} &rarr; {ss_to_industry[ss]}")

fig_sankey = go.Figure(
    go.Sankey(
        arrangement="snap",
        node={
            "label": node_labels,
            "color": node_colors,
            "customdata": node_customdata,
            "hovertemplate": "%{customdata}<extra></extra>",
            "pad": 10,
            "thickness": 14,
            "line": {"color": "#333", "width": 0.4},
        },
        link={
            "source": sources,
            "target": targets,
            "value": values,
            "color": link_colors,
            "customdata": link_hover,
            "hovertemplate": "%{customdata}<extra></extra>",
        },
    )
)
fig_sankey.update_layout(
    title=(f"Themes &rarr; Subsector &rarr; Sector &rarr; Supersector &rarr; Industry "
           f"(top {TOP_K_SECTORS} sectors, cos &ge; {TAU_SHOW}). Click any node to highlight its full path."),
    height=max(800, 24 * (n_themes + len(active_sub))),
    font={"size": 11},
)

# --- Click-to-highlight JS (highlights upstream + downstream from clicked node) ---
_js_payload = _json.dumps({
    "sources": list(map(int, sources)),
    "targets": list(map(int, targets)),
    "linkColors": list(link_colors),
    "linkHiColors": list(link_hi_colors),
    "linkKinds": list(link_kinds),
    "nodeColors": list(node_colors),
    "nThemes": n_themes,
})
SANKEY_CLICK_JS = (
    "<script>(function(){\n"
    f"  const D = {_js_payload};\n"
    "  function cascadeDown(node, nodes, links) {\n"
    "    for (let i = 0; i < D.sources.length; i++) {\n"
    "      if (D.linkKinds[i] === 'cascade' && D.sources[i] === node) {\n"
    "        links.add(i); nodes.add(D.targets[i]); cascadeDown(D.targets[i], nodes, links);\n"
    "      }\n"
    "    }\n"
    "  }\n"
    "  function cascadeUp(node, nodes, links) {\n"
    "    for (let i = 0; i < D.sources.length; i++) {\n"
    "      if (D.linkKinds[i] === 'cascade' && D.targets[i] === node) {\n"
    "        links.add(i); nodes.add(D.sources[i]); cascadeUp(D.sources[i], nodes, links);\n"
    "      }\n"
    "    }\n"
    "  }\n"
    "  function themesInto(node, nodes, links) {\n"
    "    for (let i = 0; i < D.sources.length; i++) {\n"
    "      if (D.linkKinds[i] === 'theme' && D.targets[i] === node) {\n"
    "        links.add(i); nodes.add(D.sources[i]);\n"
    "      }\n"
    "    }\n"
    "  }\n"
    "  function pathFromTheme(themeIdx) {\n"
    "    const nodes = new Set([themeIdx]), links = new Set();\n"
    "    for (let i = 0; i < D.sources.length; i++) {\n"
    "      const k = D.linkKinds[i];\n"
    "      if ((k === 'theme' || k === 'novel') && D.sources[i] === themeIdx) {\n"
    "        links.add(i); nodes.add(D.targets[i]);\n"
    "        if (k === 'theme') cascadeDown(D.targets[i], nodes, links);\n"
    "      }\n"
    "    }\n"
    "    return {nodes, links};\n"
    "  }\n"
    "  function pathFromLink(linkIdx) {\n"
    "    const s = D.sources[linkIdx], t = D.targets[linkIdx];\n"
    "    const k = D.linkKinds[linkIdx];\n"
    "    const nodes = new Set([s, t]), links = new Set([linkIdx]);\n"
    "    if (k === 'theme') { cascadeDown(t, nodes, links); }\n"
    "    else if (k === 'cascade') { cascadeDown(t, nodes, links); cascadeUp(s, nodes, links); themesInto(s, nodes, links); }\n"
    "    return {nodes, links};\n"
    "  }\n"
    "  function pathFromNode(nodeIdx) {\n"
    "    if (nodeIdx < D.nThemes) return pathFromTheme(nodeIdx);\n"
    "    const nodes = new Set([nodeIdx]), links = new Set();\n"
    "    themesInto(nodeIdx, nodes, links);\n"
    "    cascadeDown(nodeIdx, nodes, links);\n"
    "    cascadeUp(nodeIdx, nodes, links);\n"
    "    return {nodes, links};\n"
    "  }\n"
    "  function applyColors(div, lc, nc) {\n"
    "    div.data[0].link.color = lc; div.data[0].node.color = nc; Plotly.react(div, div.data, div.layout);\n"
    "  }\n"
    "  function paint(div, sel) {\n"
    "    const lc = D.linkColors.map((c, i) => sel.links.has(i) ? (D.linkHiColors[i] || c) : 'rgba(0,0,0,0.03)');\n"
    "    const nc = D.nodeColors.map((c, i) => sel.nodes.has(i) ? c : 'rgba(210,210,210,0.20)');\n"
    "    applyColors(div, lc, nc);\n"
    "  }\n"
    "  function attach() {\n"
    "    const div = document.getElementById('theme-sankey');\n"
    "    if (!div || !div.on || !div.data) { setTimeout(attach, 100); return; }\n"
    "    let activeKey = null;\n"
    "    div.on('plotly_click', function(ev) {\n"
    "      const pt = ev.points && ev.points[0]; if (!pt) return;\n"
    "      let key, sel;\n"
    "      if (pt.source !== undefined && pt.target !== undefined) {\n"
    "        const linkIdx = (pt.pointNumber !== undefined) ? pt.pointNumber : pt.index;\n"
    "        if (linkIdx === undefined) return;\n"
    "        key = 'L:' + linkIdx; sel = pathFromLink(linkIdx);\n"
    "      } else {\n"
    "        const nodeIdx = (pt.pointNumber !== undefined) ? pt.pointNumber : pt.index;\n"
    "        if (nodeIdx === undefined) return;\n"
    "        key = 'N:' + nodeIdx; sel = pathFromNode(nodeIdx);\n"
    "      }\n"
    "      if (key === activeKey) {\n"
    "        applyColors(div, D.linkColors.slice(), D.nodeColors.slice()); activeKey = null; return;\n"
    "      }\n"
    "      activeKey = key; paint(div, sel);\n"
    "    });\n"
    "  }\n"
    "  if (document.readyState === 'loading') { document.addEventListener('DOMContentLoaded', attach); }\n"
    "  else { attach(); }\n"
    "})();</script>"
)

# --- Combined HTML: Sankey + per-theme drill-down ---

parts = [
    "<html><head><meta charset='utf-8'><title>Themes v0</title>"
    "<style>"
    "body{font-family:-apple-system,Segoe UI,sans-serif;max-width:1200px;margin:24px auto;padding:0 16px;color:#222;}"
    "h1{margin-bottom:4px;}"
    "h2{margin-top:36px;border-bottom:1px solid #eee;padding-bottom:4px;}"
    "h3{margin-top:24px;}"
    ".meta{color:#666;font-size:13px;}"
    ".aligned{color:#1a7f37;font-weight:600;}"
    ".novel{color:#bf5700;font-weight:600;}"
    "li{margin:4px 0;}"
    "code{background:#f3f3f3;padding:1px 4px;border-radius:3px;}"
    "</style></head><body>",
    "<h1>Themes v0</h1>",
    f"<p class='meta'>{DATE_START[:10]} → {DATE_END[:10]} · "
    f"{len(norm_headlines):,} headlines · {len(themes)} themes · "
    f"embedder <code>{EMBEDDING_MODEL}</code> · top-{TOP_K_SECTORS} sectors · τ_show={TAU_SHOW} · τ_cov={TAU_COV}</p>",
    fig_sankey.to_html(full_html=False, include_plotlyjs="cdn", div_id="theme-sankey"),
    SANKEY_CLICK_JS,
    "<h2>Per-theme drill-down</h2>",
]

for _, row in themes.iterrows():
    t = int(row["topic"])
    top_idx = theme_top_idx[t][:TOP_HEADLINES_PER_THEME]
    cls = "aligned" if row["bucket"] == "aligned" else "novel"
    parts.append(
        f"<h3>Theme {t} — <span class='{cls}'>{row['bucket']}</span></h3>"
        f"<p class='meta'>{row['n_docs']:,} headlines · nearest sector "
        f"<b>{html_lib.escape(row['nearest_sector'])}</b> "
        f"({html_lib.escape(row['nearest_industry'])}) · cos={row['max_cos']:.2f}<br>"
        f"keywords: <code>{html_lib.escape(row['keywords'])}</code></p><ul>"
    )
    for i in top_idx:
        parts.append(f"<li>{html_lib.escape(norm_headlines[i])}</li>")
    parts.append("</ul>")
parts.append("</body></html>")

OUTPUT_HTML.write_text("\n".join(parts), encoding="utf-8")
print(f"Wrote {OUTPUT_HTML}")
fig_sankey.show()

Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/repo/notebooks/output/themes_v0.html
